**Supply Chain Analysis Project**

Hi, and welcome to my supply chain analysis project. In this project, I will focus on addressing a key business question and developing a data-driven solution. The question at hand is:

“We are experiencing issues within our supply chain, can you help us identify the problems and suggest areas for improvement?”

To explore this, I am using a fictional dataset sourced from Kaggle:
https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis

The dataset represents a global company called DataCo, and due to its international scope, it captures the complexity of a real-world supply chain. Through this analysis, I aim to uncover insights into potential challenges, risks, and improvement opportunities within the logistics and delivery processes.

In [16]:
# -------- Supply_chain_analysis_project---------
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)
from tabulate import tabulate
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

**LOADING DATA**

In [17]:
# LOADING DATA

supplychain = pd.read_csv("C:/Users/maxsh/OneDrive/Documents/Supply_Chain_Dataset/DataCoSupplyChainDataset.csv", encoding='latin-1' )


**DATA CLEANING**

In [18]:
# DATA CLEANING -----
# Checking for missing values
missing_summary = supplychain.isnull().sum().reset_index()
missing_summary.columns = ['column', 'missing_count']

print(tabulate(missing_summary, headers='keys', tablefmt='psql'))

# Dropping columns with lots of missing values.
supplychain = supplychain.drop(['Product Description', 'Order Zipcode'], axis=1)

# Converting orders into a datetime format
supplychain['order date (DateOrders)'] = pd.to_datetime(supplychain['order date (DateOrders)'])
supplychain['shipping date (DateOrders)'] = pd.to_datetime(supplychain['shipping date (DateOrders)'])

+----+-------------------------------+-----------------+
|    | column                        |   missing_count |
|----+-------------------------------+-----------------|
|  0 | Type                          |               0 |
|  1 | Days for shipping (real)      |               0 |
|  2 | Days for shipment (scheduled) |               0 |
|  3 | Benefit per order             |               0 |
|  4 | Sales per customer            |               0 |
|  5 | Delivery Status               |               0 |
|  6 | Late_delivery_risk            |               0 |
|  7 | Category Id                   |               0 |
|  8 | Category Name                 |               0 |
|  9 | Customer City                 |               0 |
| 10 | Customer Country              |               0 |
| 11 | Customer Email                |               0 |
| 12 | Customer Fname                |               0 |
| 13 | Customer Id                   |               0 |
| 14 | Customer Lname          

During the data cleaning process, it was observed that the Product Description column contained 180,519 missing values (NAs), while the Order Zipcode column had 155,679 missing values.

Given the significant proportion of missing data, both columns were deemed unsuitable for inclusion in the analysis and were subsequently removed from the dataset. While the Product Description column was unlikely to provide meaningful insights for this study, the Order Zipcode could have been valuable for a geolocation-based analysis.

In addition, the Order Date field was converted to a datetime format to facilitate potential time series analysis if the study later expands in that direction. However, the primary focus of this project remains on geographical patterns and late delivery risk.


**EXPLORATORY DATA ANALYSIS**

In [30]:
# EXPLORATORY DATA ANALYSIS -----
print("SUPPLY CHAIN DATA OVERVIEW")
print("="*60) # This creates a division. Helps data presentation.
print(f"Supply chain Shape:{supplychain.shape}") # Getting dataframe dimensions
print(f"Total Orders: {len(supplychain):,}") # Getting the length of the dataframe which will show orders
print(f"Total Columns: {supplychain.shape[1]}") # Gives the total columns

print("\n" + "="*60)
print("FIRST 5 ROWS:")
print("="*60) 
print(supplychain.head())  # Gives the first 5 rows of each column


print("\n" + "="*60)
print("SUMMARY STATISTICS:")
print("="*60)
print(supplychain.describe()) # Gives summary statistics for numerical data such as min, max, std, and count.

print("\n" + "="*60)
print("DATA TYPES:")
print("="*60)
print(tabulate(
    supplychain.dtypes.reset_index().rename(columns={'index': 'Column', 0: 'Data Type'}),
    headers='keys',
    tablefmt='psql'
))# Displays data types such as integer, float, object, etc..

print("\n" + "="*60)
print("DATASET INFO:")
print("="*60)
print(tabulate(supplychain.info(), headers='keys', tablefmt='psql')) # Displays summary of data frame

SUPPLY CHAIN DATA OVERVIEW
Supply chain Shape:(180519, 51)
Total Orders: 180,519
Total Columns: 51

FIRST 5 ROWS:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0      

Based on the data overview, the dataset contains a total of 180,519 orders. The average shipping time is approximately 3.5 days.

For this analysis, the focus will be on late delivery risk, as well as geographical location and product-related factors that may influence it. Therefore, not all columns in the dataset will be relevant for this study and some will be excluded from further analysis.

It is also worth noting that the average late delivery risk is 55%, which is relatively high and highlights the importance of exploring the underlying causes and patterns associated with delivery performance.

**PLOTLY VISUALISATIONS**



In [20]:
# PLOTLY VISUALISATIONS ------

# DELIVERY STATUS BAR CHART
print("Delivery Status Bar Chart:")
delivery_status = supplychain['Delivery Status'].value_counts().reset_index() # Counts different delivery status types
delivery_status.columns = ['Delivery Status', 'Count'] # This renames the columns in the new delivery status dataframe

delivery_status_barchart = px.bar(delivery_status,
              x='Delivery Status', # X-axis
              y='Count', # y-axis
              title='Distribution of Delivery Status', # Bar chart title
              color='Delivery Status', # Makes each bar a different colour
              text='Count', # Adds data labels
              color_discrete_sequence=px.colors.qualitative.Set2) # This is the colour palette for the bar chart
delivery_status_barchart.update_traces(textposition='outside') # positions text outside the bars
delivery_status_barchart.update_layout(showlegend=False) # Removed the legend from the graph
delivery_status_barchart.show() # Displays the delivery status bar chart

# LATE DELIVERY RISK PIE CHART
print("Late Delivery Risk Pie Chart:")

risk_counts = supplychain['Late_delivery_risk'].value_counts().reset_index() # Counts the late delivery risk types
risk_counts.columns = ['Risk', 'Count'] # renames the columns in the new dataframe
risk_counts['Risk'] = risk_counts['Risk'].map({0:'On Time', 1:'At Risk'}) # Labelling 0 as 'On time' and 1 as 'At Risk'.

late_delivery_pie = px.pie(risk_counts, # Creating a pie chart
              values = 'Count', # Adding the 0s and 1s as the data being counted towards the pie
              names = 'Risk', # Adding the risk labels to the pie
              title = 'Delivery Risk Pie Chart', # Adding the title
              color = 'Risk', # Makes each side of the pie a different colour
              color_discrete_map={'On Time': '#5AC8B1', 'At Risk': '#FF946B'}) # Adding the Hex codes to each side
late_delivery_pie.update_traces(textinfo='percent+label+value') # Adding the information to be displayed on each side of the pie
late_delivery_pie.show() # Showing figure

# SHIPPING MODE COMPARISON
print("Shipping Mode Comparison:")

shipping_mode = supplychain['Shipping Mode'].value_counts().reset_index() # Counts the different shipping modes
shipping_mode.columns = ['Shipping Mode', 'Count'] # Naming the columns in the new dataframe

shipping_mode_barchart = px.bar(shipping_mode, # Shipping mode bar chart
              x = 'Shipping Mode', # Setting X-axis
              y = 'Count', # Setting Y-axis
              title = 'Orders by shipping mode', # Adding the title
              color = 'Shipping Mode', # Colours based on the shipping mode
              text = 'Count') # Showing the count above the data bars
shipping_mode_barchart.update_traces(textposition='outside') # Displays text on top of the bars
shipping_mode_barchart.update_layout(showlegend=False) # Removing the legend from the bar chart
shipping_mode_barchart.show() # Displaying the plot

# TOP 10 PRODUCT CATEGORIES
print('Top 10 Product Categories Bar Chart:')

top_categories = supplychain['Category Name'].value_counts().head(10).reset_index() # Counts the different category names. Only using top 10.
top_categories.columns = ['Category', 'Count'] # Renaming the columns in the new dataframe

top_categories_barchart = px.bar(top_categories, # Creating the bar chart
              y='Category', # Y-axis
              x='Count',    # X-axis
              title='Top 10 Categories by Order Volume', # Adding the title
              orientation='h', # Using horizontal bars
              color='Count', # Using different colours for the count values. Will be a gradient effect
              color_continuous_scale='Blues') # Using a blue colour scale
top_categories_barchart.update_layout(yaxis={'categoryorder':'total ascending'}) # Using ascending bars
top_categories_barchart.show() # Displaying horizontal bar chart

# CUSTOMER SEGMENT DISTRIBUTION
print("Customer Segment chart:")
segment_counts = supplychain['Customer Segment'].value_counts().reset_index() # Counts the different customer segments
segment_counts.columns = ['Segment', 'Count'] # Naming the columns in the new dataframe

segment_counts_pie = px.pie(segment_counts, # Creating the pie chart
              values='Count', # Using the segment counts for the pie values
              names='Segment', # Naming each pie slice
              title='Customer Segment Distribution', # Adding the pie title
              hole=0.4)
segment_counts_pie.update_traces(textinfo='percent+label') # Adding labels to the pie chart
segment_counts_pie.show() # Displaying the chart

Delivery Status Bar Chart:


Late Delivery Risk Pie Chart:


Shipping Mode Comparison:


Top 10 Product Categories Bar Chart:


Customer Segment chart:


1. Distribution of Delivery Status
The delivery status distribution shows a high number of late deliveries, with 98,977 shipments arriving late. This figure is more than double the count of advance shipments (41,592) and significantly higher than both on-time deliveries (32,196) and cancelled orders (7,754).
This indicates a major challenge in meeting delivery deadlines, suggesting potential inefficiencies or capacity issues within the supply chain.

2. Delivery Risk
The delivery risk pie chart reveals that 54.8% of shipments are at risk of delay, compared to 45.2% expected to arrive on time. The high proportion of at-risk deliveries reinforces the earlier observation that the logistics network struggles to maintain timely performance.

3. Orders by Shipping Mode
Most shipments were sent via Standard Class (107,752), followed by Second Class (35,216), First Class (27,814), and Same Day (9,737).
This distribution suggests that customers or businesses prioritize cost efficiency over speed, though it may contribute to the overall delay rate seen in earlier charts.

4. Top 10 Categories by Order Name
‘Cleats’ were the most ordered item with 24,511 orders, followed by Men’s Footwear (22,245) and Women’s Apparel (21,035). Interestingly, Electronics ranked last with 3,156 orders, while Fishing equipment had 17,325 orders, suggesting that the dataset may reflect a niche or sport-oriented customer base rather than a general retail environment.

5. Customer Segment Distribution
The customer segment distribution indicates that 51.8% of orders came from Consumers, while 30.4% came from Corporate clients. This suggests that most orders were likely residential rather than business-related, which may also influence shipping expectations and delays.

Overall Insight

The EDA highlights significant logistics and supply chain challenges, particularly around timely delivery. Despite a large consumer base and high order volume in certain categories, the prevalence of late or at-risk deliveries suggests a need for process optimisation, improved shipment tracking, or enhanced coordination with carriers.	


**DEEP DIVE ANALYSIS**


In [21]:
# STATE-LEVEL RISK ANALYSIS
state_risk = supplychain.groupby('Customer State').agg({
    'Late_delivery_risk': 'mean', 'Order Id': 'count'}).reset_index() # Aggregating the data using functions 'mean' and 'count'.

state_risk.columns = ['State', 'Risk_Rate', 'Order_Count'] # Naming the new columns

# Convert risk rate to percentage
state_risk['Risk_Percentage'] = state_risk['Risk_Rate'] * 100

# Create choropleth map (This will create a heatmap of the USA)
state_level_risk_choropleth = px.choropleth(state_risk,
                     locations='State', # Adding each state to the displayed locations
                     locationmode='USA-states',
                     color='Risk_Percentage', # Adding colour depending on the level of risk
                     hover_data=['Order_Count'], # Adding data to the interactive elements of the map
                     color_continuous_scale=['green', 'yellow', 'red'], # Adding the risk level colour scale
                     scope='usa', # Displaying the map scope
                     title='Late Delivery Risk Rate by State (%)', # Adding a title to the map
                     labels={'Risk_Percentage': 'Risk %'}) # Adding map labels
state_level_risk_choropleth.show() # Displaying the map

# Sample data for scatter plot
sample_size = 20000
geo_data = supplychain.sample(n=min(sample_size, len(supplychain)))

# SCATTER MAP - Individual orders coloured by risk
late_delivery_scatter = px.scatter_geo(geo_data,
                      lat='Latitude', # Adding Latitude and longitude
                      lon='Longitude',
                      color='Late_delivery_risk', # Displaying colours depending on late delivery risk
                      hover_data=['Customer State', 'Customer City', 'Delivery Status'], # Adding data to the interactive elements
                      title='Delivery Risk by Location (Sample)', # Adding a title.
                      color_continuous_scale=['green', 'yellow', 'red'], # Adding the colour scale
                      scope='usa') # Scope of the map
late_delivery_scatter.show() # Displaying graph

# CITY-LEVEL ANALYSIS
city_analysis = supplychain.groupby(['Customer City', 'Customer State', 'Latitude', 'Longitude']).agg({
    'Late_delivery_risk': 'mean',
    'Order Id': 'count'
}).reset_index() # Grouping columns into a new dataframe whilst using an aggregation to apply functions to the dataframe
city_analysis.columns = ['City', 'State', 'Lat', 'Lon', 'Risk_Rate', 'Order_Count'] # Renaming the columns in the new dataframe

# Filter to cities with significant orders
city_analysis = city_analysis[city_analysis['Order_Count'] > 100]

city_analysis_scatter = px.scatter_geo(city_analysis,
                      lat='Lat', # Adding latitude and longitude
                      lon='Lon',
                      color='Risk_Rate', # Adding colour depending on the risk rate
                      size='Order_Count', # Size of the scatter dots will change depending on the number of orders
                      hover_data=['City', 'State', 'Order_Count'], # Adding data for the interactive elements of the graph
                      title='Delivery Risk by Major Cities', # Adding Graph title
                      color_continuous_scale=['green', 'yellow', 'red'], # Colour scheme
                      scope='usa', # Scope of graph
                      size_max=30)
city_analysis_scatter.show() # Displaying graph

# ===== GEOGRAPHIC SUMMARY STATISTICS =====

print("\n" + "="*60)
print("GEOGRAPHIC PERFORMANCE SUMMARY")
print("="*60)
print(tabulate(state_risk.sort_values('Risk_Percentage'), headers='keys', tablefmt='psql'))
print("\n")
print("Highest Risk State:", state_risk.loc[state_risk['Risk_Percentage'].idxmax(), 'State'],
      f"({state_risk['Risk_Percentage'].max():.1f}%)") # Takes the highest risk state percentage and rounds it to one decimal place
print("Lowest Risk State:", state_risk.loc[state_risk['Risk_Percentage'].idxmin(), 'State'],
      f"({state_risk['Risk_Percentage'].min():.1f}%)") # Takes the lowest risk state percentage and rounds it to one decimal place
print("Average State Risk:", f"{state_risk['Risk_Percentage'].mean():.1f}%") # Displays the average risk for all states
print("Total States:", state_risk['State'].nunique())  # Displays the total number of states
print("\nMost Orders:", state_risk.loc[state_risk['Order_Count'].idxmax(), 'State'],
      f"({state_risk['Order_Count'].max():,} orders)") # Displays the state with the most orders
print("="*60)


GEOGRAPHIC PERFORMANCE SUMMARY
+----+---------+-------------+---------------+-------------------+
|    | State   |   Risk_Rate |   Order_Count |   Risk_Percentage |
|----+---------+-------------+---------------+-------------------|
|  0 | 91732   |    0        |             1 |            0      |
| 25 | MT      |    0.45977  |            87 |           45.977  |
|  2 | AL      |    0.485714 |            35 |           48.5714 |
|  1 | 95758   |    0.5      |             2 |           50      |
|  6 | CO      |    0.506792 |          1914 |           50.6792 |
| 14 | ID      |    0.51497  |           167 |           51.497  |
|  4 | AZ      |    0.519498 |          3026 |           51.9498 |
| 44 | WI      |    0.521176 |           850 |           52.1176 |
| 28 | NJ      |    0.525541 |          3191 |           52.5541 |
| 24 | MO      |    0.526588 |          1354 |           52.6588 |
| 19 | LA      |    0.527426 |           948 |           52.7426 |
| 22 | MI      |    0.535226 |

**Geolocational Analysis**

1. Late Delivery Risk Rate by State
The heat map of late delivery risk by state highlights varying levels of risk across the United States. Most states show a high delivery risk rate, generally above 50%. The only exceptions are Montana (46%) and Alabama (48.5%), which appear slightly less at risk. Overall, the visualisation suggests that delivery challenges are widespread across the country, with no region performing significantly better than others.

2. Delivery Risk Scatter Map
The scatter map displays delivery risk across locations, with green points indicating low or no risk and red points indicating high risk. Although this visualisation uses only a sample of the dataset, clusters of delivery points are noticeably denser in Western regions (e.g., California) and Eastern regions (e.g., New York). This likely reflects higher order volumes in these populated areas. While no strong regional trend in risk level is apparent, the density of deliveries aligns with major population and commercial hubs.

3. City-Level Delivery Risk
At the city level, the analysis focuses on the top 100 cities by order volume. The pattern mirrors that of the scatter map with higher delivery density in both coastal regions. Interestingly, Western cities (such as those in California) show more orange hues, indicating moderately higher risk, while Eastern cities (such as those in New York) show more yellow and green tones, reflecting relatively lower risk levels.
This variation suggests that delivery reliability may differ between cities, even when overall state-level risk appears similar. Further investigation into urban logistics factors (e.g., traffic congestion, warehouse distribution, or carrier coverage) could provide deeper insight into these differences.

Overall Insight
The geospatial analysis reveals that delivery risk is a nationwide issue, though minor regional and city-level variations exist. Densely populated urban areas show higher delivery activity, with some variation in risk intensity between the East and West coasts. These findings indicate potential value in exploring regional supply chain efficiency and urban logistics optimisation as next steps.


In [22]:
# SHIPPING MODE DEEP DIVE

shipping_reliability = supplychain.groupby('Shipping Mode').agg({'Late_delivery_risk': 'mean',
'Order Id': 'count', 'Days for shipping (real)': 'mean'}).reset_index() # Grouping columns into new dataframe and using aggregate functions

shipping_reliability.columns = ['Shipping Mode', 'Risk_Rate', 'Order_Count', 'Avg_Days'] # Renaming columns

shipping_reliability['Risk_Percentage'] = shipping_reliability['Risk_Rate'] * 100 # Converting risk rate into percentage

shipping_reliability_barchart = px.bar(shipping_reliability,
              x='Shipping Mode',
              y='Risk_Percentage',
              title='Late Delivery Risk Rate by Shipping Mode',
              color='Risk_Percentage',
              color_continuous_scale=['green', 'yellow', 'red'],
              text='Risk_Percentage',
              hover_data=['Order_Count', 'Avg_Days'])
shipping_reliability_barchart.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
shipping_reliability_barchart.update_layout(yaxis_title='Risk Rate (%)')
shipping_reliability_barchart.show()

shipping_reliability_scatter = px.scatter(shipping_reliability,
                  x='Avg_Days',
                  y='Risk_Percentage',
                  size='Order_Count',
                  color='Shipping Mode',
                  title='Shipping Mode: Delivery Time vs Risk Rate',
                  labels={'Avg_Days': 'Average Delivery Days',
                          'Risk_Percentage': 'Risk Rate (%)'},
                  hover_data=['Order_Count'],
                  size_max=50)
shipping_reliability_scatter.add_hline(y=shipping_reliability['Risk_Percentage'].mean(), # Adding an average line
               line_dash="dash",
               line_color="red",
               annotation_text="Average Risk Rate")
shipping_reliability_scatter.show()

# COST vs PERFORMANCE - Using Sales as proxy for cost
shipping_performance = supplychain.groupby('Shipping Mode').agg({
    'Late_delivery_risk': 'mean',
    'Sales': 'mean',
    'Order Profit Per Order': 'mean',
    'Order Id': 'count'
}).reset_index()
shipping_performance.columns = ['Shipping Mode', 'Risk_Rate', 'Avg_Sales', 'Avg_Profit', 'Order_Count']
shipping_performance['Risk_Percentage'] = shipping_performance['Risk_Rate'] * 100

shipping_performance_scatter = px.scatter(shipping_performance,
                  x='Avg_Profit',
                  y='Risk_Percentage',
                  size='Order_Count',
                  color='Shipping Mode',
                  title='Shipping Mode: Profitability vs Delivery Risk',
                  labels={'Avg_Profit': 'Average Profit per Order ($)',
                          'Risk_Percentage': 'Risk Rate (%)'},
                  hover_data=['Avg_Sales', 'Order_Count'],
                  size_max=50)
# Adding quadrant lines
shipping_performance_scatter.add_hline(y=shipping_performance['Risk_Percentage'].mean(), # Horizontal line
               line_dash="dash", line_color="gray", opacity=0.5)
shipping_performance_scatter.add_vline(x=shipping_performance['Avg_Profit'].mean(), # Vertical line
               line_dash="dash", line_color="gray", opacity=0.5)
shipping_performance_scatter.add_annotation(x=shipping_performance['Avg_Profit'].max(), # Adding annotation text to graph
                    y=shipping_performance['Risk_Percentage'].min(),
                    text="Best: High Profit, Low Risk", showarrow=False,
                    bgcolor="lightgreen", opacity=0.7)
shipping_performance_scatter.show()


# DELIVERY STATUS BREAKDOWN BY SHIPPING MODE
status_by_mode = supplychain.groupby(['Shipping Mode', 'Delivery Status']).size().reset_index(name='Count')

status_by_mode_barchart = px.bar(status_by_mode,
              x='Shipping Mode',
              y='Count',
              color='Delivery Status',
              title='Delivery Status Breakdown by Shipping Mode',
              barmode='stack',
              text='Count')
status_by_mode_barchart.update_traces(texttemplate='%{text:.2s}', textposition='inside')
status_by_mode_barchart.show()

# Print summary insights
print("\n" + "="*60)
print("SHIPPING MODE PERFORMANCE SUMMARY")
print("="*60)
print(tabulate(shipping_reliability.sort_values('Risk_Percentage'), headers='keys', tablefmt='psql'))
print("\n")
print("Most Reliable:", shipping_reliability.loc[shipping_reliability['Risk_Percentage'].idxmin(), 'Shipping Mode'])
print("Least Reliable:", shipping_reliability.loc[shipping_reliability['Risk_Percentage'].idxmax(), 'Shipping Mode'])
print("Fastest:", shipping_reliability.loc[shipping_reliability['Avg_Days'].idxmin(), 'Shipping Mode'])
print("="*60)



SHIPPING MODE PERFORMANCE SUMMARY
+----+-----------------+-------------+---------------+------------+-------------------+
|    | Shipping Mode   |   Risk_Rate |   Order_Count |   Avg_Days |   Risk_Percentage |
|----+-----------------+-------------+---------------+------------+-------------------|
|  3 | Standard Class  |    0.380717 |        107752 |   3.99591  |           38.0717 |
|  1 | Same Day        |    0.45743  |          9737 |   0.478279 |           45.743  |
|  2 | Second Class    |    0.766328 |         35216 |   3.99083  |           76.6328 |
|  0 | First Class     |    0.953225 |         27814 |   2        |           95.3225 |
+----+-----------------+-------------+---------------+------------+-------------------+


Most Reliable: Standard Class
Least Reliable: First Class
Fastest: Same Day


**SHIPPING MODE ANALYSIS**

1. Late Delivery Risk Rate by Shipping Mode

The Late Delivery Risk Rate by Shipping Mode bar chart reveals that First Class deliveries are the most at risk, with approximately 95.3% of shipments likely to be late. Second Class also shows a high-risk rate of 76.6%, while Same Day Delivery has a lower but still notable 45.7%. Standard Class, at 38.1%, appears to be the most reliable shipping method.
Overall, the results suggest that faster shipping options such as First and Second Class may be underperforming in terms of timeliness, whereas Standard Class achieves a better balance between speed and reliability.

2. Delivery Time vs. Risk Rate

The Delivery Time vs. Risk Rate scatter plot reinforces the differences in performance between shipping modes. The visualisation makes average delivery times more explicit on the x-axis, allowing for easier comparison.
*	Standard and Second-Class deliveries have similar average delivery durations, yet Second Class faces a significantly higher lateness risk.
*	Same Day Delivery appears as expected near the lower left of the chart, showing a low average delivery time and a moderate risk level.
*	First Class shipments display a shorter average delivery time but a very high-risk rate, indicating that faster promised delivery times may not be consistently met.

This chart suggests that tighter delivery windows may correlate with increased risk of delay, potentially due to logistical constraints.

3. Profitability vs. Delivery Risk

The Profitability vs. Delivery Risk graph provides an interesting contrast between financial performance and reliability across shipping modes.

* First Class shipments, while highly profitable (≈$23 profit per order), also exhibit the highest risk levels. With approximately 27814 sales, this represents a trade-off between revenue and reliability.
*	Standard Class achieves a strong balance, combining high sales volume and solid profitability (~$22 per order) with lower delivery risk.
*	Second Class and Same Day Delivery show slightly lower profit margins (~$21–$20.8 per order) and varying degrees of risk.

Overall, Standard Class appears to be the most efficient shipping mode, offering the best balance between profit and performance. Further analysis into why First-Class shipments is both highly profitable and high risk could identify operational improvements or product-level influences.

4. Delivery Status by Shipping Mode

The Delivery Status by Shipping Mode chart shows that while Standard Class has a high number of late deliveries (≈41,000), this is largely due to its much higher order volume. Notably, Standard Class also records around 42,000 early (advance) deliveries, making it the only shipping mode with frequent early arrivals.
This suggests that Standard Class’s broader delivery window provides greater flexibility and efficiency in meeting or exceeding delivery expectations. In contrast, First Class and Second-Class methods show poorer performance, with higher proportions of delayed shipments and fewer early deliveries.

Overall Insight

The analysis highlights a clear trade-off between delivery speed, reliability, and profitability.

*	First Class: High profit, high risk.
*	Second Class: Moderate profit, high risk.
*	Standard Class: Balanced profit, lower risk — the most efficient mode overall.
*	Same Day: Low profit but reasonably low risk given its constraints.

From a supply chain management perspective, these results suggest focusing on optimising the faster shipping modes (First and Second Class), possibly through process adjustments, better forecasting, or prioritisation systems, while maintaining Standard Class as a stable and dependable delivery option.


In [23]:
# ===== PRODUCT CATEGORY ANALYSIS =====

# CATEGORY RISK ANALYSIS
category_analysis = supplychain.groupby('Category Name').agg({
    'Late_delivery_risk': 'mean',
    'Order Id': 'count',
    'Days for shipping (real)': 'mean',
    'Product Price': 'mean',
    'Order Profit Per Order': 'mean'
}).reset_index()
category_analysis.columns = ['Category', 'Risk_Rate', 'Order_Count', 'Avg_Days', 'Avg_Price', 'Avg_Profit']
category_analysis['Risk_Percentage'] = category_analysis['Risk_Rate'] * 100

# Sort for better display
category_analysis = category_analysis.sort_values('Risk_Percentage', ascending=True)

# Risk by category
category_analysis_barchart = px.bar(category_analysis,
              y='Category',  # Category on y-axis
              x='Risk_Percentage',  # Risk on x-axis
              title='Late Delivery Risk Rate by Product Category',
              orientation='h',
              color='Risk_Percentage',
              color_continuous_scale=['green', 'yellow', 'red'],
              text='Risk_Percentage',
              hover_data=['Order_Count', 'Avg_Days'])
category_analysis_barchart.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
category_analysis_barchart.update_layout(
    xaxis_title='Risk Rate (%)',
    yaxis_title='Category',
    height=600  # Making it taller for better readability
)
category_analysis_barchart.show()

# SCATTER PLOT
category_analysis_scatter = px.scatter(category_analysis,
                  x='Avg_Price',
                  y='Risk_Percentage',
                  size='Order_Count',
                  color='Category',
                  title='Product Category: Price vs Delivery Risk',
                  labels={'Avg_Price': 'Average Product Price ($)',
                          'Risk_Percentage': 'Risk Rate (%)'},
                  hover_data=['Avg_Days', 'Order_Count'],
                  trendline='ols',  # Add line of best fit
                  size_max=40)
category_analysis_scatter.show()

# SCATTER PLOT - Shipping Days vs Risk with Trendline
category_analysis_scatter_2 = px.scatter(category_analysis,
                  x='Avg_Days',
                  y='Risk_Percentage',
                  size='Order_Count',
                  color='Category',
                  title='Product Category: Delivery Time vs Risk Rate',
                  labels={'Avg_Days': 'Average Delivery Days',
                          'Risk_Percentage': 'Risk Rate (%)'},
                  hover_data=['Avg_Price', 'Order_Count'],
                  trendline='ols',  # Add line of best fit
                  size_max=40)
category_analysis_scatter_2.show()


# ===== CATEGORY SUMMARY STATISTICS =====
print("\n" + "="*60)
print("PRODUCT CATEGORY PERFORMANCE SUMMARY")
print("="*60)
print(tabulate(category_analysis.sort_values('Risk_Percentage', ascending=False), headers='keys', tablefmt='psql'))
print("\n")
print("Highest Risk Category:", category_analysis.loc[category_analysis['Risk_Percentage'].idxmax(), 'Category'],
      f"({category_analysis['Risk_Percentage'].max():.1f}%)")
print("Lowest Risk Category:", category_analysis.loc[category_analysis['Risk_Percentage'].idxmin(), 'Category'],
      f"({category_analysis['Risk_Percentage'].min():.1f}%)")
print("Average Category Risk:", f"{category_analysis['Risk_Percentage'].mean():.1f}%")
print("\nMost Ordered Category:", category_analysis.loc[category_analysis['Order_Count'].idxmax(), 'Category'],
      f"({category_analysis['Order_Count'].max():,} orders)")
print("Most Profitable Category:", category_analysis.loc[category_analysis['Avg_Profit'].idxmax(), 'Category'],
      f"(${category_analysis['Avg_Profit'].max():.2f} avg profit)")
print("="*60)


PRODUCT CATEGORY PERFORMANCE SUMMARY
+----+----------------------+-------------+---------------+------------+-------------+--------------+-------------------+
|    | Category             |   Risk_Rate |   Order_Count |   Avg_Days |   Avg_Price |   Avg_Profit |   Risk_Percentage |
|----+----------------------+-------------+---------------+------------+-------------+--------------+-------------------|
| 23 | Golf Bags & Carts    |    0.688525 |            61 |    3.34426 |    169.99   |     29.6733  |           68.8525 |
| 32 | Lacrosse             |    0.600583 |           343 |    3.61808 |     38.7655 |     12.7239  |           60.0583 |
| 37 | Pet Supplies         |    0.589431 |           492 |    3.38821 |     84.4    |      7.29524 |           58.9431 |
|  8 | Cameras              |    0.581081 |           592 |    3.33784 |    452.04   |     51.1652  |           58.1081 |
| 41 | Strength Training    |    0.576577 |           111 |    3.74775 |    494.554  |      2.99378 |       

**PRODUCT CATEGORY ANALYSIS**

1. Late Delivery Risk by Product Category

The Late Delivery Risk by Product Category chart reveals notable variation in delivery performance across product types.

*	Golf Bags & Carts have the highest lateness risk, with 68.9% of orders arriving late (61 total orders).
*	Lacrosse follows with a 60.1% lateness risk (343 orders), and Pet Supplies ranks third at 60% (492 orders).
*	The lowest risk category is Men’s Golf Clubs, with a 47.7% lateness rate (283 orders).

It is interesting that golf-related items appear at both extremes with one of the most delayed and one of the most reliable categories. This could suggest variations in supplier performance or inventory handling within the same product family.

2. Product Category: Price vs. Delivery Risk

The Price vs. Delivery Risk analysis shows that Garden products have the highest average price at $532, paired with a 55% lateness risk.
Most other categories cluster around an average price of $50, with delivery risk rates between 50–55%.
This suggests that price does not strongly correlate with delivery risk, meaning that both high- and low-value products experience similar delivery reliability. However, high-priced categories may deserve special attention, as delays could have a greater impact on customer satisfaction.

3. Product Category: Delivery Time vs. Risk Rate

The Delivery Time vs. Risk Rate plot shows that most product categories cluster around 3.5 average delivery days and 50–55% lateness risk.
Notable outliers include:

•	Baby Products – Average delivery time of 3.1 days with a 52.6% risk rate.
•	Strength Training Products – Average delivery time of 3.7 days with a 57.6% risk rate.

These outliers suggest that certain niche categories experience different delivery patterns, possibly due to specialized handling requirements, stock availability, or supplier location.

Overall Insight

Across product categories, delivery risk rates are relatively consistent, generally falling between 50–60%. However, certain categories such as Golf Bags & Carts and Strength Training Equipment exhibit elevated lateness, which could indicate logistical bottlenecks or supplier-related issues. Conversely, products like Men’s Golf Clubs demonstrate that category-specific improvements are achievable, even within similar product types.

Further analysis could explore whether supplier region, shipping mode, or order volume contributes most to these delivery performance differences.


**MACHINE LEARNING MODEL**

Model Selection & Why I Chose Them

For this part of the project, I used three models: Random Forest, XGBoost, and Logistic Regression. I wanted to test how different methods handled predicting whether a delivery would be on time (0) or at risk (1). Each model approaches the problem slightly differently, so this gave me a good mix to compare performance and interpretability.

Random Forest Classifier

I started with the Random Forest Classifier because it’s reliable and handles classification problems like this very well. My target variable is binary (0 or 1), so the trees don’t need to be overly complex. Random Forest builds many smaller trees and combines their results, which helps improve accuracy and reduces overfitting.
It’s also a good model to start with because it performs strongly across a range of data types and gives useful feature importance scores to understand what drives the predictions.

 XGBoost Classifier

I then used XGBoost, which is another tree-based model but uses boosting to improve prediction accuracy. It builds trees sequentially, where each one learns from the mistakes of the previous ones. This method can often increase performance by fine-tuning how the model learns patterns in the data.
XGBoost can take a bit longer to train compared to Random Forest, but it’s known for being highly optimized and efficient in practice. I included it to see if it could achieve slightly better predictive accuracy on this dataset.

Logistic Regression

Finally, I ran a Logistic Regression model. These works differently, instead of building trees, it estimates the probability that a delivery is “at risk” based on a weighted combination of the input features.
I included it because it’s fast, easy to interpret, and offers a nice comparison to the more complex tree-based models. While it doesn’t produce feature importances in the same way, the coefficients still show which features have the strongest positive or negative influence on delivery risk.


In [24]:
# Feature list
features = [
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Benefit per order',
    'Sales per customer',
    'Product Price',
    'Customer Segment',
    'Order Region',
    'Order State',
    'Shipping Mode',
    'Product Category Id',
    'Department Name',
    'Market'
]

target = 'Late_delivery_risk'

x = supplychain[features]
y = supplychain[target]

# Handle missing values

x = x.fillna(0)
y = y.fillna(0)

# Encode categorical variables
x = pd.get_dummies(x, drop_first = True)

# Train - Test Split
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest

rfmodel = RandomForestClassifier(
    n_estimators=300,      # a good boost in accuracy vs 100-200
    max_depth=15,          # prevents trees from overgrowing
    min_samples_split=4,   # makes splits more efficient
    min_samples_leaf=2,    # avoids very small leaf nodes
    n_jobs=-1,             # use all CPU cores
    random_state=42,
    verbose=1              # show progress in VS Code terminal
)

rfmodel.fit(X_train, y_train)
# Evaluate Model
y_pred = rfmodel.predict(X_test)
y_proba = rfmodel.predict_proba(X_test)[:, 1]
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    5.7s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:   33.1s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:   51.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:    0.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.1s


Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.86      0.92     16308
           1       0.89      1.00      0.94     19796

    accuracy                           0.94     36104
   macro avg       0.95      0.93      0.93     36104
weighted avg       0.94      0.94      0.93     36104

ROC-AUC Score: 0.9720046454458484


[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:    0.2s finished


**Random Forest Model Summary**

For the Random Forest Classifier, the model performed very well across both classes.
For class ‘0’ (On Time), it achieved a precision of 1.00, recall of 0.86, and an F1-score of 0.92, with 16,308 samples in this group.
For class ‘1’ (At Risk), the model scored 0.89 precision, 1.00 recall, and an F1-score of 0.94, with 19,796 samples.

This shows that the Random Forest was particularly precise when predicting deliveries that would not arrive late, while still performing very strongly in detecting those that were at risk.
The model achieved an overall accuracy of 94% and an excellent ROC-AUC score of 0.972, demonstrating strong discriminatory power between the two classes.
In total, the Random Forest completed 300 tasks during training, running efficiently while maintaining high predictive performance.

In [25]:
# Import
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Train Model
xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=1
)

xgb_model.fit(X_train, y_train)

# Evaluate
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\nXGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba_xgb))



XGBoost Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97     16308
           1       0.96      1.00      0.98     19796

    accuracy                           0.97     36104
   macro avg       0.98      0.97      0.97     36104
weighted avg       0.98      0.97      0.97     36104

ROC-AUC Score: 0.9793129403605767


**XGBoost Model Summary**

For the XGBoost Classifier, the model achieved excellent performance across both classes.
For class ‘0’ (No Risk), it scored a precision of 1.00, recall of 0.94, and an F1-score of 0.97, with 16,308 samples.
For class ‘1’ (At Risk), the model achieved a precision of 0.96, recall of 1.00, and an F1-score of 0.98, with 19,796 samples.

Overall, the XGBoost model achieved an accuracy of 97% and a ROC-AUC score of 0.979.
These results are very strong and slightly higher than the Random Forest model, showing that XGBoost handled the prediction task with a high level of accuracy and consistency.

In [26]:
# Import
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale Data 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Train Model
lr_model = LogisticRegression(
    max_iter=200,
    solver='saga',   # faster for large datasets
    n_jobs=-1,       # use all CPU cores
    verbose=1        # show progress
)
lr_model.fit(X_train_lr, y_train_lr)

# Evaluate
y_pred_lr = lr_model.predict(X_test_lr)
y_proba_lr = lr_model.predict_proba(X_test_lr)[:, 1]

print("\nLogistic Regression Classification Report:\n", classification_report(y_test_lr, y_pred_lr))
print("ROC-AUC Score:", roc_auc_score(y_test_lr, y_proba_lr))


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


max_iter reached after 363 seconds


c:\Users\maxsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge




Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97     16308
           1       0.96      1.00      0.98     19796

    accuracy                           0.97     36104
   macro avg       0.98      0.97      0.97     36104
weighted avg       0.97      0.97      0.97     36104

ROC-AUC Score: 0.9743899564247996


**Logistic Regression Model Summary**

For the Logistic Regression model, the results were also very strong and consistent with the tree-based models.
For class ‘0’ (No Risk), the model achieved a precision of 1.00, recall of 0.94, and an F1-score of 0.97, with 16,308 samples.
For class ‘1’ (At Risk), it scored a precision of 0.96, recall of 1.00, and an F1-score of 0.98, with 19,796 samples.

Overall, the model achieved an accuracy of 97% and a ROC-AUC score of 0.974, showing that even without using tree-based methods, logistic regression handled this binary classification problem extremely well.

**Confusion Matrix**

In [27]:
def plot_confusion_matrix_plotly(y_true, y_pred, model_name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=['Actual On Time', 'Actual Late'],
        columns=['Predicted On Time', 'Predicted Late']
    )
    
    fig = px.imshow(
        cm_df,
        text_auto=True,
        color_continuous_scale='Blues',
        title=f"Confusion Matrix - {model_name}"
    )
    fig.update_layout(xaxis_title="Predicted", yaxis_title="Actual")
    return fig

rf_matrix = plot_confusion_matrix_plotly(y_test, y_pred, "Random Forest")
rf_matrix.show()

XGB_matrix = plot_confusion_matrix_plotly(y_test, y_pred_xgb, "XGBoost")
XGB_matrix.show()

lr_matrix = plot_confusion_matrix_plotly(y_test_lr, y_pred_lr, "Logistic Regression")
lr_matrix.show()


**Random Forest Confusion Matrix**

* RF never predicted a late delivery as on time (0 false negatives), which is excellent for catching late deliveries.

* It misclassified 2,340 on-time deliveries as late (false positives). So, RF is more conservative, it prefers to predict late rather than miss a late delivery.

**XGBoost Confusion Matrix**

* XGB misclassified very few on-time deliveries (922) and almost no late deliveries as on-time (2).

* More balanced than RF: slightly higher correct on-time predictions (15,386 vs. 13,968) and almost same late prediction accuracy.

**Logistic Regression Confusion Matrix**

* LR slightly underperforms XGB: it misclassifies 42 late deliveries as on-time (false negatives), which RF avoided entirely.

* Slightly fewer correct late predictions than RF/XGB.

**Overall**

RF is safer if missing a late delivery is very costly but it overestimates late deliveries. XGB provides the best tradeoff with nearly perfect recall for late deliveries while keeping false positives low. LR is almost as good as XGB but slightly worse at catching late deliveries.

**Feature Importance**

In [28]:
def plot_feature_importance_plotly(model, feature_names, model_name="Model", top_n=15):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    top_features = importances.nlargest(top_n).sort_values()
    
    fig = px.bar(
        top_features,
        x=top_features.values,
        y=top_features.index,
        orientation='h',
        title=f"Top {top_n} Feature Importances - {model_name}",
        labels={'x': 'Importance Score', 'y': 'Feature'}
    )
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    return fig

rf_features = plot_feature_importance_plotly(rfmodel, X_train.columns, "Random Forest")
rf_features.show()

xgb_features = plot_feature_importance_plotly(xgb_model, X_train.columns, "XGBoost")
xgb_features.show()

**Feature Importance Analysis**

Random Forest Top Features:

* Days for shipping (actual) – Actual delivery duration is strongly correlated with lateness.

* Days for shipment (scheduled) – Planned shipment days help assess potential delay risk.

* Shipping Mode_Standard Class – Standard Class had the highest volume of deliveries and late deliveries, making it a key predictor.

* Lower-ranked features include geographic regions (e.g., Marche, Alagoas) and product price (8th), suggesting some influence from location and item value.

XGBoost Top Features:

* Shipping Mode_Standard Class – Most important predictor, highlighting that categorical frequency matters significantly.

* Days for shipping / scheduled days – Important, but slightly less than shipping mode.

* Order State_Marche – Appears 6th, revealing regional supply chain issues. Other states such as Alagoas, Tamaulipas, and Lombardia appeared in RF but not XGB, suggesting model-specific handling of geographic interactions.

Cross-Model Insights:

* Shipping Mode_Standard Class is a consistently important predictor for both models due to its high number of deliveries and late deliveries (41,023 out of 107,752).

* Days for shipping is universally predictive, indicating that delays in actual delivery time are the strongest indicator of lateness.

* Geographic regions play a secondary but notable role, with certain states (e.g., Marche) acting as “hotspots” for late deliveries.

* Product Price has minor predictive influence, possibly reflecting prioritization for higher-value items.

Interpretation:

* Random Forest prioritises numerical features (actual and scheduled shipping days) and treats geographic features as secondary.

* XGBoost captures more subtle interactions between shipping mode, days, and location, making it sensitive to regional supply chain patterns.

Real-World Implications:

* Standard Class deliveries are at higher risk for lateness and require closer monitoring.

* Certain states, like Marche, indicate localized supply chain inefficiencies that could be targeted for improvement.

* Monitoring planned vs. actual shipping days is critical for early prediction and intervention of late deliveries.

**ROC-AUC Scores**

In [29]:
results = {
    "Random Forest": roc_auc_score(y_test, y_proba),
    "XGBoost": roc_auc_score(y_test, y_proba_xgb),
    "Logistic Regression": roc_auc_score(y_test_lr, y_proba_lr)
}

# Convert to DataFrame
results_df = pd.DataFrame(list(results.items()), columns=["Model", "ROC-AUC Score"])

# Plot
roc_comparison = px.bar(
    results_df,
    x="Model",
    y="ROC-AUC Score",
    color="Model",
    text="ROC-AUC Score",
    title="Model ROC-AUC Comparison",
    color_discrete_sequence=["#6DC5D1", "#FFB26B", "#C1F2B0"]  # optional custom colors
)

roc_comparison.show()


**ROC-AUC Scores**

The ROC-AUC scores for all three models are very similar, with 0.972 for Random Forest, 0.979 for XGBoost, and 0.974 for Logistic Regression, indicating that each method is a viable option for predicting delivery outcomes. However, when considering the feature importance results and confusion matrices, it is clear that each model has areas where it performs particularly well. Random Forest tends to overestimate late deliveries, which could be important when shipping high-value products where on-time delivery is critical. XGBoost, on the other hand, shows greater sensitivity to shipping locations, making it especially useful for managing a global supply chain. Logistic Regression performs only slightly worse than XGBoost, and while feature importance was not analysed for this model due to its different structure, it remains a strong baseline for prediction. Overall, the choice of model may depend on the specific priorities of delivery management, whether minimizing missed late deliveries or capturing regional shipping nuances.

**Project Conclusion**

To conclude this project, I first conducted an initial EDA covering delivery status, delivery risk, shipping mode, the top 10 product categories, and customer segment distribution. This analysis highlighted some supply chain challenges related to delivery times and set the stage for a deeper dive. In the subsequent analysis, I explored geolocation delivery risk, shipping mode, and product category, which provided more in-depth insights into the supply chain.

I found it particularly interesting to examine which states had the highest delivery risk and to compare delivery time versus risk rate. For example, the risk rate for Standard Class shipments was lower when average delivery days were higher, which was a fascinating trend to observe. It was also insightful to see that Golf Bags and Carts were the most at-risk products for being late, while Men’s Golf Clubs were among the least at risk. Observing how the machine learning models leveraged different features to make predictions added another layer of understanding.

As a quick side note for the future, even though this is a fictional dataset, I wondered if products like Golf Bags and Carts might have come from the Marche supplier, given the delivery patterns observed.

**Solutions and Outcomes**

While this project did not implement operational changes to prevent late deliveries, the outcome has been highly valuable in identifying which features in the supply chain could be contributing to delays and disruptions. By combining exploratory data analysis, deep dives into geolocation, shipping modes, and product categories, and machine learning feature importance, I was able to uncover key patterns that may drive late deliveries.

For example, the analysis highlighted that Standard Class shipments and certain products like Golf Bags and Carts are at higher risk of being late. Geographic factors, such as specific states with recurring delivery issues, also surfaced as potential contributors. The feature importance results from the Random Forest and XGBoost models were particularly insightful, as they pointed out which variables the models considered most predictive of delays. These findings could guide supply chain managers on where to focus monitoring, allocate resources, or adjust processes to reduce lateness.

Ultimately, the solution this project provides is clarity and insight: it identifies the areas of the supply chain that are most likely to cause delivery disruptions. These insights could be used to inform further investigations, prioritise improvements, and support data-driven decision-making in logistics management. The project demonstrates not only the ability to analyse complex data but also to translate patterns into actionable business insights.